# EDA


### 0. Check files and read in data
- Check that the files are uploaded
- Read in the CSV

In [0]:
# Check files

dbutils.fs.ls("/Volumes/marathos/default/raw")

In [0]:
# Read in data 

DATA_PATH = "/Volumes/marathos/default/raw"

df_raw = spark.read.csv(f"{DATA_PATH}/TWO_CENTURIES_OF_UM_RACES.csv", header=True, inferSchema=True)

### 1. Initial EDA

#### 1.1. Explore columns and schema
Find out 
- number of rows
- number of columns
- check schema if you have inferred it or specify an appropriate schema

- descriptive summary of numerical fields

In [0]:

df_raw.count()
# Find out number of rows

len(df_raw.columns)
# Find out number of columns

print(f"Number of rows:{df_raw.count()}")
print(f"Number of columns:{len(df_raw.columns)}")

In [0]:
df_raw.columns
# See all column names

df_raw.printSchema()
# See all column names and types

In [0]:
df_raw.display(10)

#### Summary of data exploration

- 7 miljon rows
- 13 columns

- Columns and Schema: 
    - Year of event: integer
    - Event dates: string -> datetieme
    - Event name: string 
    - Event distance/length: string 
    - Event number of finishers: integer 
    - Athlete performance: string -> vad ska detta vara
    - Athlete club: string 
    - Athlete country: string 
    - Athlete year of birth: double -> borde vara samma som athlete year (int)
    - Athlete gender: string
    - Athlete age category: string 
    - Athlete average speed: string -> double
    - Athlete ID: integer



#### 1.2. Explore the data

- **Nulls**
A lot of nulls for teh following categorys:
    - Age of birth
    - Age category
    - Club

    - seems to be only for 2018 - how many event where there 2018?

- **Events**
    - Count of unique events = 26.900

- how are ages distributed among the runners?
- which countries are most represented?

### Unique events

# TO DO

- kanske lista antal unika event per år?
- tydlig lista på återkommande event per år?

In [0]:
from pyspark.sql.functions import countDistinct

df_raw.select("Event name").distinct().display()
# View list of all unique events 

df_raw.groupBy("Event name") \
    .agg(countDistinct("Year of event").alias("number_of_years")) \
    .orderBy("number_of_years", ascending=False) \
    .display()
# View reaccuring events

unique_events = df_raw.select("Event name").distinct().count()
print(f"Number of unique events:{unique_events}")
# Count number of unique events



### Nulls

Findings

**Nulls**

- *year of birth and age category:*
    - There are a lot of null values in these columns  - almost the same amount in both
        - Athlete year of birth', 588161
        - Athlete age category', 584938
    
# TO DO
- find connection between null values and years
-  Räkna nulls per event-typ — är det en specifik sport?
df_raw_nulls.groupBy("Event name").count().orderBy("count", ascending=False).display()

- 3. Kombinera år + event — se exakt var nullsen sitter
df_raw_nulls.groupBy("Year", "Event name").count().orderBy("Year", "count", ascending=False).display()

- 4. Jämför med hela datasetet — är andelen nulls högre för 2018?
# Totalt per år
df_raw.groupBy("Year").count().orderBy("Year").display()

# Nulls per år
df_raw_nulls.groupBy("Year").count().orderBy("Year").display()

- jämför om det är samma rader som är noll för age och category

- kolla vad null i resterande columner är och om vi kan få bor det



In [0]:
# Counts number of null values in each column
from pyspark.sql.functions import col, sum as spark_sum

null_count = df_raw.select(
    [spark_sum(col(columns).isNull().cast("int")).alias(columns) for columns in df_raw.columns
])

# Display columns with null values
null_count = null_count.collect()[0].asDict()
[(column, nulls) for column, nulls in null_count.items() if nulls > 0]

In [0]:

# Display rows with nulls for ´Athlete age category´
df_raw_nulls = df_raw.filter(col("Athlete age category").isNull())

df_raw_nulls.display()

In [0]:
# Find connectins between null values

# 1. Nulls per year
df_raw_nulls.groupBy("Year of event").count().orderBy("Year of event").display()